# 修補後的資格證明元件比較（CLI）

No-Merkle：公開 C 的資格檢查與封包繫結。Merkle：隱藏成員的資格證明與封包繫結。

兩組都不在本 Notebook 收票、更新 has_voted 或計票。量測不代表完整投票系統、匿名防重投或純密碼學耗時。

請重新啟動核心後依序執行。預設 8 個合成樣本、2 輪，只確認介面與資料紀錄；完成後再設定正式實驗規模。


In [6]:
import base64
import csv
from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path
import platform
import statistics
import subprocess
import time

from zk_normal import User
import zk_no_merkle_bound as nm_module
import zk_merkle as m_module
from zk_no_merkle_bound import ZKVotingSystem as NoMerkleSystem
from zk_merkle import ZKMerkleVotingSystem, ZKMerkleTree, MerkleContext, MerkleProofPacket

PROJECT = Path(m_module.__file__).resolve().parent
if Path.cwd().resolve() != PROJECT:
    raise RuntimeError(f"請將 Notebook 工作目錄設為專案根目錄：{PROJECT}")
if not hasattr(ZKMerkleVotingSystem, 'verifyBallotProof'):
    raise RuntimeError('載入了舊 zk_merkle；請重新啟動核心')
print('已載入選票繫結版 No-Merkle 與私密成員 Merkle。')


已載入選票繫結版 No-Merkle 與私密成員 Merkle。


In [7]:
# 先確認流程；不要在尚未成功前啟動完整效能實驗。
# TEST_AMOUNTS = [8]
# ROUNDS = 2

# 正式測試時再自行改為：
TEST_AMOUNTS = [8, 16, 32, 64, 128, 256]
ROUNDS = 6

# None 使用 build_circuit.py 成功後記錄的 latest.json。
# 如需固定版本，改成該次 run_... 目錄的絕對或相對路徑。
MERKLE_CIRCUITS_DIR = None
NO_MERKLE_ARTIFACT_DIR = None


## 計時定義

- `prove_call_s`：generateVoteProof 的完整呼叫，包含 witness、暫存檔與 Node/CLI。
- `verify_call_s`：同一次資格檢查內 verifyZKProof 的耗時，包含驗證所需檔案與 Node/CLI。
- `validation_call_s`：資格資訊比對、封包雜湊檢查及 verifyZKProof 的完整耗時。

validation 包含 verify，不要把兩個數字相加。驗證只呼叫一次。逐筆 CSV 不保存身分、秘密、C、binding、證明或封包。

註冊承諾準備及建樹時間另存 preparation.csv。Merkle 保留原本的完整建樹方式，未使用功能測試的稀疏捷徑。記憶體此次不列為效能指標。


In [8]:
def validate_no_merkle(system, approved_commitments, packet):
    # 比較用的假名制資格驗證介面；不使用具身分參數的 castVote。
    signals = packet.public_signals
    if not nm_module.valid_signals(signals):
        raise ValueError('INVALID_PUBLIC_SIGNALS')
    if signals[0] not in approved_commitments:
        raise ValueError('COMMITMENT_NOT_REGISTERED')
    if signals[2] != nm_module.packet_hash(packet.encrypted_vote):
        raise ValueError('VOTE_HASH_MISMATCH')
    if not system.verifyZKProof(packet.proof, signals):
        raise ValueError('ZK_VERIFICATION_FAILED')
    return True


def timed_validation(system, validate):
    # 在實際那次驗證外加計時；無論成功失敗都還原物件方法。
    original = system.verifyZKProof
    previous_instance_value = system.__dict__.get('verifyZKProof')
    had_instance_value = 'verifyZKProof' in system.__dict__
    durations = []
    def measured(*args, **kwargs):
        start = time.perf_counter()
        try:
            return original(*args, **kwargs)
        finally:
            durations.append(time.perf_counter() - start)
    system.verifyZKProof = measured
    try:
        start = time.perf_counter()
        accepted = validate()
        total = time.perf_counter() - start
    finally:
        if had_instance_value:
            system.verifyZKProof = previous_instance_value
        else:
            del system.verifyZKProof
    if accepted is not True or len(durations) != 1:
        raise RuntimeError('資格驗證未成功，或密碼驗證呼叫次數不是一次')
    return durations[0], total


def describe(values):
    return {'mean': statistics.mean(values), 'median': statistics.median(values),
            'stdev': statistics.stdev(values) if len(values) > 1 else 0.0,
            'min': min(values), 'max': max(values)}


def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


In [9]:
def run_experiment():
    if type(ROUNDS) is not int or ROUNDS < 1:
        raise ValueError('ROUNDS 必須為正整數')
    if not TEST_AMOUNTS or any(type(n) is not int or n < 2 for n in TEST_AMOUNTS):
        raise ValueError('TEST_AMOUNTS 必須是至少 2 的整數')

    nm = NoMerkleSystem(NO_MERKLE_ARTIFACT_DIR)
    merkle = ZKMerkleVotingSystem(MERKLE_CIRCUITS_DIR)
    depths = {n: m_module.calculate_optimal_depth(n) for n in TEST_AMOUNTS}
    for n, depth in depths.items():
        if depth not in merkle.depths:
            raise ValueError(f'N={n} 需要 h={depth}，但指定版本尚未建置')
        for path in merkle._get_paths(depth):
            if not path.is_file():
                raise FileNotFoundError(path)
    for path in [nm.artifact_dir / 'final.zkey', Path(nm.vkey_path),
                 nm.artifact_dir / 'no_merkle_bound_js/no_merkle_bound.wasm']:
        if not path.is_file():
            raise FileNotFoundError(path)
    if nm_module.packet_hash('MQ==') != m_module.packet_hash('MQ=='):
        raise RuntimeError('兩組封包雜湊定義不一致')

    run_id = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    folder = PROJECT / 'research_results/bound_component_benchmark' / run_id
    folder.mkdir(parents=True, exist_ok=False)
    metadata = {
        'status': 'running', 'run_id': run_id,
        'started_utc': datetime.now(timezone.utc).isoformat(),
        'platform': platform.platform(), 'python': platform.python_version(),
        'node': subprocess.check_output(['node', '--version'], text=True).strip(),
        'rounds': ROUNDS, 'sample_counts': TEST_AMOUNTS,
        'source': 'synthetic users; alternating choices encoded in Base64; NOT AES',
        'scope': 'stateless eligibility and packet-binding proof components, CLI',
        'order': 'odd rounds NM then M; even rounds M then NM',
        'warmup': 'none; first sample included',
        'merkle_artifacts': str(merkle.circuits_dir),
        'no_merkle_artifacts': str(nm.artifact_dir),
        'files_sha256': {},
        'memory_measured': False,
    }
    provenance = [PROJECT / 'benchmark.ipynb', Path(nm_module.__file__),
                  Path(m_module.__file__), PROJECT / 'zk_no_merkle.py',
                  PROJECT / 'zk_normal.py', PROJECT / 'commitment.js',
                  PROJECT / 'package-lock.json',
                  nm.artifact_dir / 'final.zkey', Path(nm.vkey_path),
                  nm.artifact_dir / 'no_merkle_bound_js/no_merkle_bound.wasm',
                  merkle.circuits_dir / 'results.json']
    for depth in sorted(set(depths.values())):
        provenance.extend(merkle._get_paths(depth))
    for path in provenance:
        if path.is_file():
            metadata['files_sha256'][str(path)] = file_hash(path)
    manifest = folder / 'environment.json'
    def save_metadata():
        manifest.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
    save_metadata()
    raw_fields = ['run_id', 'round', 'n_samples', 'depth', 'scheme', 'scheme_position',
                  'sample_index', 'prove_call_s', 'verify_call_s', 'validation_call_s']
    metrics = ['prove_call_s', 'verify_call_s', 'validation_call_s']
    summary_fields = ['run_id', 'round', 'n_samples', 'depth', 'scheme', 'scheme_position'] + [
        f'{metric}_{stat}' for metric in metrics for stat in ['mean', 'median', 'stdev', 'min', 'max']]
    prep_fields = ['run_id', 'round', 'n_samples', 'depth', 'commitments_prepare_s',
                   'nm_registry_prepare_s', 'merkle_tree_build_s']
    summary_rows = []
    try:
        with (folder / 'per_vote.csv').open('w', newline='', encoding='utf-8') as raw_file, \
             (folder / 'round_summary.csv').open('w', newline='', encoding='utf-8') as summary_file, \
             (folder / 'preparation.csv').open('w', newline='', encoding='utf-8') as prep_file:
            raw_writer = csv.DictWriter(raw_file, fieldnames=raw_fields)
            summary_writer = csv.DictWriter(summary_file, fieldnames=summary_fields)
            prep_writer = csv.DictWriter(prep_file, fieldnames=prep_fields)
            for writer in [raw_writer, summary_writer, prep_writer]:
                writer.writeheader()
            for round_no in range(1, ROUNDS + 1):
                order = ['NM', 'M'] if round_no % 2 else ['M', 'NM']
                for n in TEST_AMOUNTS:
                    depth = depths[n]
                    print(f'Round {round_no}/{ROUNDS}, N={n}, h={depth}, order={order}', flush=True)
                    # Both schemes use the same synthetic users/secrets/packets within a pair.
                    samples = [(User(f'BENCH_{i}'), nm.generateVoterSecret(),
                                base64.b64encode(str(1 + i % 2).encode()).decode()) for i in range(n)]
                    started = time.perf_counter()
                    commitments = [nm.computeIdentityCommitment(voter, secret)
                                   for voter, secret, _ in samples]
                    commitments_s = time.perf_counter() - started
                    started = time.perf_counter()
                    approved = frozenset(commitments)
                    nm_registry_s = time.perf_counter() - started
                    if len(approved) != n:
                        raise RuntimeError('Duplicate synthetic commitment')
                    started = time.perf_counter()
                    tree = ZKMerkleTree(commitments, depth)
                    context = MerkleContext(tree.get_root(), depth)
                    tree_s = time.perf_counter() - started
                    prep_writer.writerow(dict(run_id=run_id, round=round_no, n_samples=n, depth=depth,
                        commitments_prepare_s=commitments_s, nm_registry_prepare_s=nm_registry_s,
                        merkle_tree_build_s=tree_s))
                    prep_file.flush()
                    for position, scheme in enumerate(order, 1):
                        measurements = {metric: [] for metric in metrics}
                        for index, (voter, secret, encrypted) in enumerate(samples):
                            # Path retrieval is preparation, outside the proof-call timer.
                            path = tree.get_path(index) if scheme == 'M' else None
                            started = time.perf_counter()
                            if scheme == 'NM':
                                data = nm.generateVoteProof(voter, secret, encrypted)
                            else:
                                data = merkle.generateVoteProof(voter, secret, path,
                                        context.root, context.depth, encrypted)
                            prove_s = time.perf_counter() - started
                            packet = MerkleProofPacket(encrypted, data['proof'], data['public_signals'])
                            if scheme == 'NM':
                                verify_s, validation_s = timed_validation(nm,
                                    lambda: validate_no_merkle(nm, approved, packet))
                            else:
                                verify_s, validation_s = timed_validation(merkle,
                                    lambda: merkle.verifyBallotProof(context, packet))
                            measured = dict(prove_call_s=prove_s, verify_call_s=verify_s,
                                            validation_call_s=validation_s)
                            raw_writer.writerow(dict(run_id=run_id, round=round_no, n_samples=n,
                                depth=depth, scheme=scheme, scheme_position=position,
                                sample_index=index, **measured))
                            raw_file.flush()
                            for metric, value in measured.items():
                                measurements[metric].append(value)
                        row = dict(run_id=run_id, round=round_no, n_samples=n, depth=depth,
                                   scheme=scheme, scheme_position=position)
                        for metric, values in measurements.items():
                            for stat, value in describe(values).items():
                                row[f'{metric}_{stat}'] = value
                        summary_writer.writerow(row)
                        summary_file.flush()
                        summary_rows.append(row)
                        print(f"  {scheme}: prove={row['prove_call_s_mean']:.4f}s, "
                              f"verify={row['verify_call_s_mean']:.4f}s, "
                              f"validation={row['validation_call_s_mean']:.4f}s", flush=True)
        # Compare round means: don't pretend all votes from all rounds are independent repetitions.
        aggregate = []
        for n in TEST_AMOUNTS:
            for scheme in ['NM', 'M']:
                selected = [r for r in summary_rows if r['n_samples'] == n and r['scheme'] == scheme]
                aggregate.append({'n_samples': n, 'depth': depths[n], 'scheme': scheme,
                    'completed_rounds': len(selected), 'round_means': {
                        metric: describe([r[f'{metric}_mean'] for r in selected]) for metric in metrics}})
        (folder / 'across_rounds.json').write_text(json.dumps(aggregate, indent=2), encoding='utf-8')
        metadata['status'] = 'completed'
        print(f'BOUND_COMPONENT_RUN_OK\nResults: {folder}')
    except Exception as error:
        metadata['status'] = 'failed'
        metadata['error_type'] = type(error).__name__
        metadata['error'] = str(error)
        print(f'執行失敗，已完成的逐筆資料保留於：{folder}')
        raise
    finally:
        metadata['ended_utc'] = datetime.now(timezone.utc).isoformat()
        save_metadata()
    return folder


In [10]:
result_folder = run_experiment()


Round 1/6, N=8, h=3, order=['NM', 'M']
  NM: prove=0.7162s, verify=0.4893s, validation=0.4893s
  M: prove=0.8762s, verify=0.5496s, validation=0.5497s
Round 1/6, N=16, h=4, order=['NM', 'M']
  NM: prove=0.8227s, verify=0.5391s, validation=0.5392s
  M: prove=1.0467s, verify=0.6154s, validation=0.6155s
Round 1/6, N=32, h=5, order=['NM', 'M']
  NM: prove=0.8296s, verify=0.5425s, validation=0.5426s
  M: prove=0.9950s, verify=0.5759s, validation=0.5760s
Round 1/6, N=64, h=6, order=['NM', 'M']
  NM: prove=0.9107s, verify=0.5900s, validation=0.5901s
  M: prove=1.2623s, verify=0.6816s, validation=0.6817s
Round 1/6, N=128, h=7, order=['NM', 'M']
  NM: prove=0.7704s, verify=0.4982s, validation=0.4983s
  M: prove=0.9930s, verify=0.5176s, validation=0.5176s
Round 1/6, N=256, h=8, order=['NM', 'M']
  NM: prove=0.7470s, verify=0.4886s, validation=0.4886s
  M: prove=1.0292s, verify=0.5306s, validation=0.5306s
Round 2/6, N=8, h=3, order=['M', 'NM']
  M: prove=0.8167s, verify=0.4840s, validation=0.4841s

## 結果與限制

- `per_vote.csv`：逐筆耗時，不含身分或證明內容。
- `round_summary.csv`：各輪、各組的平均、中位數、標準差與範圍。
- `across_rounds.json`：跨輪平均值的分布，正式比較應保留輪次結構。
- `preparation.csv`：承諾準備、No-Merkle 查表集合準備及完整 Merkle 建樹成本。
- `environment.json`：參數、執行狀態、版本路徑與程式/產物雜湊。

此版本沒有記憶體測量，也沒有常駐 API 比較。身分隱藏限定於 Merkle 的證明與驗證介面，測試程序仍持有合成 witness；同秘密同封包的 binding 相同，不宣稱跨提交不可連結。兩组不具相同公開資訊，正是研究要描述的差異。
